# DeepEval MCP Metrics - Excel Trace Evaluation

This notebook only evaluates MCP trace data from an Excel workbook. The MCP server and trace generator run outside this notebook. The Excel file is the handoff artifact.

## 1. Install Dependencies

In [ ]:
%pip install deepeval mcp openai pandas openpyxl

## 2. Imports

In [ ]:
import json
from pathlib import Path

import pandas as pd

from deepeval.metrics.mcp.mcp_task_completion import MCPTaskCompletionMetric
from deepeval.metrics.mcp.multi_turn_mcp_use_metric import MultiTurnMCPUseMetric
from deepeval.metrics.mcp_use_metric.mcp_use_metric import MCPUseMetric
from deepeval.models.llms.azure_model import AzureOpenAIModel
from deepeval.test_case import ConversationalTestCase, LLMTestCase, MCPServer, MCPToolCall, Turn
from mcp.types import CallToolResult, TextContent, Tool

## 3. Judge Model Credentials

Set the judge model credentials before reading or evaluating the Excel trace.

In [ ]:
AZURE_OPENAI_API_KEY = "PASTE_AZURE_OPENAI_API_KEY_HERE"
AZURE_OPENAI_ENDPOINT = "https://PASTE_RESOURCE_NAME.openai.azure.com"
AZURE_OPENAI_DEPLOYMENT_NAME = "PASTE_AZURE_OPENAI_DEPLOYMENT_NAME_HERE"
AZURE_OPENAI_MODEL_NAME = "gpt-5"
AZURE_OPENAI_API_VERSION = "PASTE_AZURE_OPENAI_API_VERSION_HERE"

AZURE_OPENAI_COST_PER_INPUT_TOKEN = 0.0
AZURE_OPENAI_COST_PER_OUTPUT_TOKEN = 0.0

judge_model = AzureOpenAIModel(
    model=AZURE_OPENAI_MODEL_NAME,
    deployment_name=AZURE_OPENAI_DEPLOYMENT_NAME,
    api_key=AZURE_OPENAI_API_KEY,
    base_url=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
    cost_per_input_token=AZURE_OPENAI_COST_PER_INPUT_TOKEN,
    cost_per_output_token=AZURE_OPENAI_COST_PER_OUTPUT_TOKEN,
)

## 4. Excel File

Run the next cell to upload `Live_Dummy_MCP_Traces.xlsx` in Colab. If you are running locally and the file is already beside this notebook, the same cell will use that local file instead.

In [ ]:
TRACE_FILE = Path("Live_Dummy_MCP_Traces.xlsx")

try:
    from google.colab import files

    print("Upload Live_Dummy_MCP_Traces.xlsx")
    uploaded = files.upload()

    if uploaded:
        uploaded_name = next(iter(uploaded.keys()))
        TRACE_FILE = Path(uploaded_name)
except ImportError:
    print("Not running in Colab. Using local Excel file path.")

if not TRACE_FILE.exists():
    raise FileNotFoundError(
        "Live_Dummy_MCP_Traces.xlsx was not found. Upload the generated Excel file or place it beside this notebook."
    )

TRACE_FILE

## 5. Read Generated Sheets

In [ ]:
input_cases = pd.read_excel(TRACE_FILE, sheet_name="input_cases")
tool_definitions = pd.read_excel(TRACE_FILE, sheet_name="tool_definitions")
llm_outputs = pd.read_excel(TRACE_FILE, sheet_name="llm_outputs")
tool_calls = pd.read_excel(TRACE_FILE, sheet_name="tool_calls")
deepeval_objects = pd.read_excel(TRACE_FILE, sheet_name="deepeval_objects")

llm_outputs

## 6. Inspect Tool Definitions

In [ ]:
pd.set_option("display.max_colwidth", 3000)

tool_definitions[["tool_name", "tool_object"]]

## 7. Inspect Generated DeepEval Objects

In [ ]:
deepeval_objects[["case_id", "dummy_mcp_server_object", "llm_test_case_object"]]

## 8. MCP Server Definition For DeepEval

In [ ]:
dummy_support_server = MCPServer(
    server_name="dummy_support",
    transport="streamable-http",
    available_tools=[
        Tool(
            name="get_customer_profile",
            description="Get a customer profile by customer ID.",
            inputSchema={
                "type": "object",
                "properties": {"customer_id": {"type": "string"}},
                "required": ["customer_id"],
                "additionalProperties": False,
            },
        ),
        Tool(
            name="get_order_status",
            description="Get order status and order details by order ID.",
            inputSchema={
                "type": "object",
                "properties": {"order_id": {"type": "string"}},
                "required": ["order_id"],
                "additionalProperties": False,
            },
        ),
        Tool(
            name="search_policy_docs",
            description="Search support policy documents.",
            inputSchema={
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
                "additionalProperties": False,
            },
        ),
        Tool(
            name="calculate_refund",
            description="Calculate refund eligibility for an order.",
            inputSchema={
                "type": "object",
                "properties": {
                    "order_id": {"type": "string"},
                    "reason": {"type": "string"},
                },
                "required": ["order_id", "reason"],
                "additionalProperties": False,
            },
        ),
        Tool(
            name="create_support_ticket",
            description="Create a support ticket for a customer issue.",
            inputSchema={
                "type": "object",
                "properties": {
                    "customer_id": {"type": "string"},
                    "issue_summary": {"type": "string"},
                    "priority": {"type": "string", "enum": ["low", "normal", "high"]},
                },
                "required": ["customer_id", "issue_summary", "priority"],
                "additionalProperties": False,
            },
        ),
    ],
)

## 9. Test Case Shapes

`MCPUseMetric` evaluates single-turn `LLMTestCase` objects. `MultiTurnMCPUseMetric` and `MCPTaskCompletionMetric` evaluate `ConversationalTestCase` objects built from the same Excel trace.

All positive and negative cases are read from the Excel file.

## 10. Convert Excel Rows Into DeepEval Test Cases

In [ ]:
def make_call_tool_result(result_json):
    result_dict = json.loads(result_json)
    return CallToolResult(
        content=[TextContent(type="text", text=json.dumps(result_dict, ensure_ascii=False))],
        structuredContent={"result": result_dict},
        isError=False,
    )


def make_mcp_tool_call(row):
    return MCPToolCall(
        name=row["tool_name"],
        args=json.loads(row["tool_arguments_json"]),
        result=make_call_tool_result(row["tool_result_json"]),
    )


test_cases = []
conversational_test_cases = []
evaluation_case_rows = []

for _, output_row in llm_outputs.iterrows():
    case_tool_rows = tool_calls[tool_calls["case_id"] == output_row["case_id"]]
    mcp_tool_calls = [make_mcp_tool_call(tool_row) for _, tool_row in case_tool_rows.iterrows()]

    expected_label = "Positive"
    if "expected_label" in llm_outputs.columns and pd.notna(output_row.get("expected_label")):
        expected_label = str(output_row.get("expected_label"))

    test_cases.append(
        LLMTestCase(
            input=str(output_row["user_input"]),
            actual_output=str(output_row["actual_output"]),
            expected_outcome=str(output_row["expected_outcome"]),
            mcp_servers=[dummy_support_server],
            mcp_tools_called=mcp_tool_calls,
        )
    )

    conversational_test_cases.append(
        ConversationalTestCase(
            name=str(output_row["case_id"]),
            scenario=str(output_row["user_input"]),
            expected_outcome=str(output_row["expected_outcome"]),
            mcp_servers=[dummy_support_server],
            turns=[
                Turn(role="user", content=str(output_row["user_input"])),
                Turn(
                    role="assistant",
                    content="I will use the available MCP tools to complete the request.",
                    mcp_tools_called=mcp_tool_calls,
                ),
                Turn(role="assistant", content=str(output_row["actual_output"])),
            ],
        )
    )

    evaluation_case_rows.append(
        {
            "case_id": output_row["case_id"],
            "expected_label": expected_label,
        }
    )

len(test_cases), len(conversational_test_cases), pd.DataFrame(evaluation_case_rows)

## 11. Run MCPUseMetric

In [ ]:
mcp_use_rows = []

for test_case, case_row in zip(test_cases, evaluation_case_rows):
    metric = MCPUseMetric(model=judge_model, threshold=0.5, include_reason=True, async_mode=False, verbose_mode=True)
    score = metric.measure(test_case)
    mcp_use_rows.append(
        {
            "case_id": case_row["case_id"],
            "expected_label": case_row["expected_label"],
            "metric": "MCPUseMetric",
            "score": round(score, 3),
            "success": metric.is_successful(),
            "reason": metric.reason,
        }
    )

mcp_use_report = pd.DataFrame(mcp_use_rows)
mcp_use_report

## 12. Run MultiTurnMCPUseMetric

In [ ]:
multi_turn_use_rows = []

for conversational_case, case_row in zip(conversational_test_cases, evaluation_case_rows):
    metric = MultiTurnMCPUseMetric(model=judge_model, threshold=0.5, include_reason=True, async_mode=False, verbose_mode=True)
    score = metric.measure(conversational_case)
    multi_turn_use_rows.append(
        {
            "case_id": case_row["case_id"],
            "expected_label": case_row["expected_label"],
            "metric": "MultiTurnMCPUseMetric",
            "score": round(score, 3),
            "success": metric.is_successful(),
            "reason": metric.reason,
        }
    )

multi_turn_use_report = pd.DataFrame(multi_turn_use_rows)
multi_turn_use_report

## 13. Run MCPTaskCompletionMetric

In [ ]:
task_completion_rows = []

for conversational_case, case_row in zip(conversational_test_cases, evaluation_case_rows):
    metric = MCPTaskCompletionMetric(model=judge_model, threshold=0.5, include_reason=True, async_mode=False, verbose_mode=True)
    score = metric.measure(conversational_case)
    task_completion_rows.append(
        {
            "case_id": case_row["case_id"],
            "expected_label": case_row["expected_label"],
            "metric": "MCPTaskCompletionMetric",
            "score": round(score, 3),
            "success": metric.is_successful(),
            "reason": metric.reason,
        }
    )

mcp_task_completion_report = pd.DataFrame(task_completion_rows)
mcp_task_completion_report

## 14. Final Report

In [ ]:
final_report = pd.concat(
    [mcp_use_report, multi_turn_use_report, mcp_task_completion_report],
    ignore_index=True,
)

final_report

## 15. Export Final Report

This cell writes the metric results to Excel. In Colab, it also starts a browser download.

In [ ]:
REPORT_FILE = Path("DeepEval_MCP_Final_Report.xlsx")

with pd.ExcelWriter(REPORT_FILE, engine="openpyxl") as writer:
    final_report.to_excel(writer, sheet_name="final_report", index=False)
    mcp_use_report.to_excel(writer, sheet_name="mcp_use_metric", index=False)
    multi_turn_use_report.to_excel(writer, sheet_name="multi_turn_mcp_use", index=False)
    mcp_task_completion_report.to_excel(writer, sheet_name="mcp_task_completion", index=False)

print(f"Saved report to {REPORT_FILE}")

try:
    from google.colab import files

    files.download(str(REPORT_FILE))
except ImportError:
    from IPython.display import FileLink, display

    display(FileLink(str(REPORT_FILE)))